In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
BASE_DIR = '../Fruits_data/train'
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 30  # Slightly higher for fine-tuning
MODEL_PATH = '../MLMODELS/fruits_veg_identify.keras'

In [ ]:
for cls in os.listdir(BASE_DIR):
    n_images = len(os.listdir(os.path.join(BASE_DIR, cls)))
    print(f"{cls}: {n_images} images")

apple: 68 images
banana: 75 images
beetroot: 88 images
bell pepper: 90 images
cabbage: 92 images
capsicum: 89 images
carrot: 82 images
cauliflower: 79 images
chilli pepper: 87 images
corn: 87 images
cucumber: 94 images
eggplant: 84 images
garlic: 92 images
ginger: 68 images
grapes: 100 images
jalepeno: 88 images
kiwi: 88 images
lemon: 82 images
lettuce: 97 images
mango: 86 images
onion: 94 images
orange: 69 images
paprika: 83 images
pear: 89 images
peas: 100 images
pineapple: 99 images
pomegranate: 79 images
potato: 77 images
raddish: 81 images
soy beans: 97 images
spinach: 97 images
sweetcorn: 91 images
sweetpotato: 69 images
tomato: 92 images
turnip: 98 images
watermelon: 84 images


In [ ]:
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.1,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range =[0.7,1.3],
    fill_mode='nearest'
)

In [ ]:
train_generator = datagen.flow_from_directory(
    BASE_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='training',
    class_mode='categorical'
)

validation_generator = datagen.flow_from_directory(
    BASE_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation',
    class_mode='categorical'
)

Found 2823 images belonging to 36 classes.
Found 292 images belonging to 36 classes.


In [ ]:
# Save class labels
labels = '\n'.join(sorted(train_generator.class_indices.keys()))
with open('labels.txt', 'w') as f:
    f.write(labels)

In [ ]:
y = train_generator.classes
classes = np.unique(y)
class_weights = compute_class_weight('balanced', classes=classes, y=y)
class_weights = dict(enumerate(class_weights))

In [ ]:
IMAGE_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, 3)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SHAPE,
    include_top=False,
    weights='imagenet'
)
# Freeze all layers except last 30 for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

In [ ]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(train_generator.class_indices), activation='softmax')
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 36)             │         9,252 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,595,172 (9.90 MB)

 Trainable params: 2,192,292 (8.36 MB)

 Non-trainable params: 402,880 (1.54 MB)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

In [ ]:
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True)
]

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

c:\Users\gcbho\OneDrive\Desktop\CAPSTONE\zipcartprototype\ZipCartPrototype\backend\pythonservices\.venv\Lib\site-packages\PIL\Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 132s 3s/step - accuracy: 0.1392 - loss: 3.3619 - val_accuracy: 0.4075 - val_loss: 2.4624
Epoch 2/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 124s 3s/step - accuracy: 0.4042 - loss: 2.2870 - val_accuracy: 0.5377 - val_loss: 1.5984
Epoch 3/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 123s 3s/step - accuracy: 0.5530 - loss: 1.6314 - val_accuracy: 0.5993 - val_loss: 1.3020
Epoch 4/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 142s 3s/step - accuracy: 0.6277 - loss: 1.2705 - val_accuracy: 0.6610 - val_loss: 1.0768
Epoch 5/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 129s 3s/step - accuracy: 0.6876 - loss: 1.0966 - val_accuracy: 0.6884 - val_loss: 0.9654
Epoch 6/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 106s 2s/step - accuracy: 0.7287 - loss: 0.9509 - val_accuracy: 0.6884 - val_loss: 0.9387
Epoch 7/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.7506 - loss: 0.8458 - val_accuracy: 0.7192 - val_loss: 0.8487
Epoch 8/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 134s 3s/step - accuracy: 0.7623 - loss: 0.7825 - val_accuracy: 0.7226 - v

In [ ]:
model.save(MODEL_PATH)

In [ ]:
def predict_image(model, img_path, target_size=(IMAGE_SIZE, IMAGE_SIZE), threshold=0.6):
    """
    Returns the predicted class and confidence.
    If confidence < threshold, returns 'Unknown'.
    """
    try:
        img = image.load_img(img_path, target_size=target_size)
        img_array = np.expand_dims(image.img_to_array(img)/255.0, axis=0)
        preds = model.predict(img_array)
        class_idx = np.argmax(preds)
        class_labels = list(train_generator.class_indices.keys())
        confidence = float(np.max(preds))
        if confidence < threshold:
            return "Unknown", confidence
        return class_labels[class_idx], confidence
    except Exception as e:
        return f"Error: {e}", 0.0

In [ ]:
img_path = '../trialImages/apple.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Prediction: potato (65.42%)


In [ ]:
img_path = '../trialImages/banana.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Prediction: banana (99.96%)


In [ ]:
img_path = '../trialImages/beetroot.png'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Prediction: beetroot (99.99%)


In [ ]:
img_path = '../trialImages/Bell Pepper.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Prediction: Unknown (48.56%)


In [ ]:
img_path = '../trialImages/cabbage.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Prediction: cabbage (99.99%)


In [ ]:
img_path = '../trialImages/capsicum.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
Prediction: Unknown (53.81%)


In [ ]:
img_path = '../trialImages/carrot.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Prediction: carrot (94.30%)


In [ ]:
img_path = '../trialImages/cauliflower.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Prediction: cauliflower (99.98%)


In [ ]:
img_path = '../trialImages/chilli peper.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Prediction: chilli pepper (100.00%)


In [ ]:
img_path = '../trialImages/corn.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Prediction: Unknown (55.90%)


In [ ]:
img_path = '../trialImages/cucumber.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Prediction: cucumber (100.00%)


In [ ]:
img_path = '../trialImages/egg plant.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
Prediction: eggplant (99.80%)


In [ ]:
img_path = '../trialImages/garlic.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Prediction: garlic (95.62%)


In [ ]:
img_path = '../trialImages/ginger.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
Prediction: ginger (97.76%)


In [ ]:
img_path = '../trialImages/grapes.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Prediction: grapes (99.99%)


In [ ]:
img_path = '../trialImages/jalapeno.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Prediction: chilli pepper (85.21%)


In [ ]:
img_path = '../trialImages/kiwi.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Prediction: pear (64.67%)


In [ ]:
img_path = '../trialImages/lemon.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Prediction: Unknown (48.56%)


In [ ]:
img_path = '../trialImages/lettuce.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Prediction: lettuce (96.76%)


In [ ]:
img_path = '../trialImages/mango.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
Prediction: orange (99.93%)


In [ ]:
img_path = '../trialImages/onion.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Prediction: onion (99.99%)


In [ ]:
img_path = '../trialImages/orange.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Prediction: lemon (63.50%)


In [ ]:
img_path = '../trialImages/paprika.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
Prediction: capsicum (93.39%)


In [ ]:
img_path = '../trialImages/pear.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Prediction: mango (92.54%)


In [ ]:
img_path = '../trialImages/peas.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Prediction: cucumber (99.14%)


In [ ]:
img_path = '../trialImages/pineapple.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Prediction: pineapple (100.00%)


In [ ]:
img_path = '../trialImages/pomogranate.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Prediction: Unknown (44.18%)


In [ ]:
img_path = '../trialImages/potato.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Prediction: potato (98.59%)


In [ ]:
img_path = '../trialImages/soybeans.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Prediction: soy beans (100.00%)


In [ ]:
img_path = '../trialImages/spinach.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
Prediction: spinach (95.24%)


In [ ]:
img_path = '../trialImages/strawberry.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Prediction: pomegranate (75.67%)


In [ ]:
img_path = '../trialImages/sweetcorn.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Prediction: sweetcorn (88.40%)


In [ ]:
img_path = '../trialImages/sweetpotato.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Prediction: sweetpotato (97.88%)


In [ ]:
img_path = '../trialImages/tomato.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Prediction: tomato (99.81%)


In [ ]:
img_path = '../trialImages/turnip.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Prediction: raddish (100.00%)


In [ ]:
img_path = '../trialImages/watermelon.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Prediction: watermelon (99.93%)


In [ ]:
img_path = '../trialImages/image1.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
Prediction: apple (78.63%)


In [ ]:
img_path = '../trialImages/image2.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Prediction: banana (97.19%)


In [ ]:
img_path = '../trialImages/image3.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Prediction: orange (68.98%)


In [ ]:
img_path = '../trialImages/image4.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Prediction: pineapple (99.65%)


In [ ]:
img_path = '../trialImages/apple_bulk (1).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Prediction: Unknown (59.07%)


In [ ]:
img_path = '../trialImages/apple_bulk (10).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Prediction: sweetpotato (75.64%)


In [ ]:
img_path = '../trialImages/apple_bulk (14).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Prediction: Unknown (22.27%)


In [ ]:
img_path = '../trialImages/apple_bulk (7).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Prediction: mango (66.59%)


In [ ]:
img_path = '../trialImages/banana (11).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Prediction: banana (99.74%)


In [ ]:
img_path = '../trialImages/banana (22).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
Prediction: banana (94.64%)


In [ ]:
img_path = '../trialImages/banana (8).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Prediction: banana (99.86%)


In [ ]:
img_path = '../trialImages/beefsteak_tomato (1).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Prediction: Unknown (48.84%)


In [ ]:
img_path = '../trialImages/beefsteak_tomato (17).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Prediction: Unknown (29.53%)


In [ ]:
img_path = '../trialImages/beefsteak_tomato (6).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Prediction: tomato (88.25%)


In [ ]:
img_path = '../trialImages/carrot (10).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
Prediction: carrot (73.17%)


In [ ]:
img_path = '../trialImages/carrot (5).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
Prediction: carrot (93.94%)


In [ ]:
img_path = '../trialImages/carrot (8).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Prediction: carrot (96.35%)


In [ ]:
img_path = '../trialImages/Image_6.png'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Prediction: chilli pepper (99.96%)


In [ ]:
img_path = '../trialImages/Image_33.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Prediction: mango (99.28%)


In [ ]:
img_path = '../trialImages/cucumber (5).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
Prediction: cucumber (99.88%)


In [ ]:
img_path = '../trialImages/cucumber (8).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Prediction: cucumber (90.07%)


In [ ]:
img_path = '../trialImages/flat_cabbage (4).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Prediction: Unknown (58.59%)


In [ ]:
img_path = '../trialImages/flat_cabbage (5).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Prediction: Unknown (27.66%)


In [ ]:
img_path = '../trialImages/flat_cabbage (6).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
Prediction: Unknown (26.83%)


In [ ]:
img_path = '../trialImages/flat_cabbage (7).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Prediction: cabbage (75.76%)


In [ ]:
img_path = '../trialImages/grapes (12).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
Prediction: Unknown (52.29%)


In [ ]:
img_path = '../trialImages/grapes (2).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Prediction: peas (61.49%)


In [ ]:
img_path = '../trialImages/grapes (7).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Prediction: Unknown (32.82%)


In [ ]:
img_path = '../trialImages/beetroot (1).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Prediction: beetroot (98.60%)


In [ ]:
img_path = '../trialImages/beetroot (3).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction: beetroot (96.82%)


In [ ]:
img_path = '../trialImages/beetroot (5).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction: beetroot (99.58%)


In [ ]:
img_path = './banana1.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: banana (99.89%)


In [ ]:
img_path = './hello.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Prediction: carrot (99.96%)
